# Kalibratie — trainingen scoren op herschrijfbaarheid

Draai de scorer op een handvol trainingen, bekijk de output, en leg naast je handmatige labels.

**Voor je begint:**
1. `pip install -r requirements.txt`
2. Zet je API-key in een `.env`-bestand (zie `.env.example`).
3. Download de Google Sheet als **xlsx** (Bestand → Downloaden) en zet het pad hieronder.
4. Zet `Scoringsrubric_herschrijfbaarheid_v1.md` in dezelfde map.

In [3]:
from dotenv import load_dotenv
load_dotenv()

import importlib, score_trainings as st
importlib.reload(st)   # zodat je edits in score_trainings.py meteen meepakt

IN_XLSX   = "/Users/hugovandenbelt/Downloads/TrainingenLijst_50.xlsx"      # <- gedownloade sheet
RUBRIC    = "Scoringsrubric_herschrijfbaarheid_v1.md"
N         = 46                                        # eerst een klein setje
WEB_SEARCH = False                                      # actualiteitscheck aan

## 1. Sheet inlezen en de content-parser controleren
Check eerst dat de kolommen goed herkend worden en dat de brontekst schoon uit de JSON komt.

In [4]:
df, cols = st.read_input(IN_XLSX)
print("Herkende kolommen:", cols)
print("Aantal rijen:", len(df))

# Inspecteer de eerste training: parsed content + geschoonde brontekst
row0 = df.iloc[0]
content0 = st.parse_content(row0[cols['content']])
print("\nJSON-keys:", list(content0))
print("Dagen:", st.extract_days(content0, row0[cols['days']] if cols['days'] else None))
print("\n--- brontekst (eerste 800 tekens) ---")
print(st.build_source_text(content0, str(row0[cols['name']]))[:800])

Herkende kolommen: {'id': 'id', 'name': 'name', 'content': 'content', 'days': None}
Aantal rijen: 46

JSON-keys: ['days', 'intro', 'setup', 'modules', 'summary', 'follow_up', 'objectives', 'certification', 'summary_edudex', 'prior_knowledge', 'target_audience']
Dagen: None

--- brontekst (eerste 800 tekens) ---
[days]
5

[intro]
Linux is open source software, wat betekent dat de onderliggende broncode vrij toegankelijk is voor gebruikers en ontwikkelaars. Vanwege het open source karakter is Linux gratis te gebruiken en wordt het vrij gedistribueerd. Denk hierbij aan de volgende distributies: Ubuntu, Debian, Fedora en Red Hat Enterprise. Linux wordt veel gebruikt als platform voor servers. Veel websites bijvoorbeeld draaien op een Linux server, in combinatie met Apache of Nginx als webserver en PHP als serverside programmeertaal. Daarnaast wordt Linux gebruikt als basissoftware voor firewalls, routers, telefoon, fileservers en nog veel meer.

Toepassingen

 De cursus Linux is bedoeld vo

## 2. Scoren
Kost een paar seconden per training (meer met web search). Output gaat ook naar een xlsx.

In [5]:
out = st.run_file(IN_XLSX, "trainingen_scored_Top50.xlsx", RUBRIC, limit=N, use_web_search=WEB_SEARCH)
out[['titel','eindscore','verdict','actualiteit_type','actualiteit_severity','menselijke_input_nodig','scorer_confidence']]

[1/46] Training Linux                                -> 62 redelijk
[2/46] Training SQL                                  -> 80 rijk
[3/46] Opleiding Vormgeven met Illustrator           -> 70 rijk
[4/46] Cursus XML                                    -> 48 redelijk
[5/46] Cursus XSL                                    -> 38 dun
[6/46] Cursus UML                                    -> 52 redelijk
[7/46] Cursus Photoshop Professional                 -> 53 redelijk
[8/46] Opleiding C# Professional                     -> 73 rijk
[9/46] Cursus LDAP                                   -> 65 rijk
[10/46] Training E-commerce                           -> 68 rijk
[11/46] Training Creative Cloud                       -> 78 rijk
[12/46] Cursus CRM                                    -> 68 rijk
[13/46] Cursus After Effects Professional             -> 60 redelijk
[14/46] Cursus Database Design                        -> 70 rijk
[15/46] Training Google Analytics 4 - GA4             -> 62 redelijk
[16/46] Cur

,titel,eindscore,verdict,actualiteit_type,actualiteit_severity,menselijke_input_nodig,scorer_confidence
0,Training Linux,62,redelijk,additief,medium,False,high
1,Training SQL,80,rijk,none,none,False,medium
2,Opleiding Vormgeven met Illustrator,70,rijk,additief,low,False,high
3,Cursus XML,48,redelijk,additief,medium,False,medium
4,Cursus XSL,38,dun,structureel,high,True,high
5,Cursus UML,52,redelijk,additief,low,False,medium
6,Cursus Photoshop Professional,53,redelijk,additief,medium,False,medium
7,Opleiding C# Professional,73,rijk,additief,low,False,high
8,Cursus LDAP,65,rijk,additief,low,False,high
9,Training E-commerce,68,rijk,additief,low,False,medium


## 3. Naast je handmatige labels leggen
Vul je eigen labels in (id → score). Kijk of de scores correleren en of de verdict-grenzen landen waar je ze wil.
Stel daarna in `score_trainings.py` de `IMPACT`-tabel of `WEIGHTS` bij en draai opnieuw (reload-cel bovenaan).

In [17]:
import pandas as pd

# jouw handmatige labels: {training_id: score}
labels = {
    # 43: 40,   # Cursus XSL
    # 87: 70,   # Cursus LDAP
}

cmp = out[['training_id','titel','eindscore','verdict']].copy()
cmp['label'] = cmp['training_id'].map(labels)
cmp['delta'] = (cmp['eindscore'] - cmp['label']).abs()
cmp.dropna(subset=['label'])

,training_id,titel,eindscore,verdict,label,delta


In [ ]:
# Verdeling van de scores over de populatie — zie hoe scheef je input is
out['verdict'].value_counts().reindex(['al_nieuwe_stijl','rijk','redelijk','dun','onbruikbaar']).fillna(0).astype(int)

## 4. Volledige feedback van één training bekijken
Handig om te zien of de vooruitgerichte feedback (bruikbaar / strippen / gaten / rewrite_guidance) klopt.

In [8]:
import textwrap
r = out.iloc[0]
for k in ['titel','kern','eindscore','verdict','actualiteit_actie','bruikbaar','strippen','gaten','rewrite_guidance']:
    print(f"{k}:")
    print(textwrap.fill(str(r[k]), 100, initial_indent='  ', subsequent_indent='  '))
    print()

titel:
  Opleiding PHP Professional

kern:
  PHP Professional leert (junior) developers programmeren in PHP, van functioneel naar object-
  georiënteerd, met MySQL-database-implementatie, en levert een zelf gebouwde webapplicatie
  (praktijkcase webwinkel) op.

eindscore:
  58

verdict:
  redelijk

actualiteit_actie:
  refresh: voeg moderne PHP-ecosysteemonderdelen toe (Composer, PSR-4 autoloading, namespaces, PHP
  8.x taalfeatures) aan de OO-module en check of MySQL-module PDO/prepared statements benoemt in
  plaats van alleen klassieke queries.

bruikbaar:
  Resultaat-sectie met expliciete leeruitkomst (zelfstandig webapplicatie bouwen) | Praktijkcase
  webwinkel als rode draad | Vier modules met concrete sub-onderwerpen: Inleiding programmeren
  (array's, regex, cookies, sessies, exception handling, debugging), MySQL, OO programmeren
  (classes/objects, exception handling, design patterns) | Security als doorlopend thema (input-
  validatie, veilige database-toegang) | Eindopdracht